In [1]:
%%capture
!pip install transformers
!pip install trl
!pip install datasets
!pip install tqdm
!pip install openpyxl --upgrade
!pip install evaluate
!pip install peft

In [ ]:
# This is the Jupyter notebook for single-task SLM with LoRA finetuning on the DICOM series harmonization task.

## Load data

In [15]:
import pandas as pd
from tqdm import tqdm
from datasets import Dataset,concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from trl import SFTTrainer
def split_data(d):
    full_dataset = d.train_test_split(test_size=0.2, shuffle=True, seed=0)
    return full_dataset['train'], full_dataset['test']

# TASK: DICOM Series Harmonization Data
df1=pd.read_csv('../ground_truth_series_description.csv')
df1['index_col'] = df1.index
d1 = Dataset.from_pandas(df1)
d1_train,d1_test=split_data(d1)

# TASK: Radiology Report Labeling Data
df2=pd.read_csv('../IU-GroundTruth.csv')
d2 = Dataset.from_pandas(df2)

# TASK: Impression Generation Data
df3=pd.read_csv('../IU-GroundTruth.csv')
df3 = df3.drop(df3[df3.REPORT.str.contains(pat="IMPRESSION")==False].index)
df3['impression']=True

#Split all data and combine (this is to make sure the single-task train test data are the same with the multi-task)
d3 = Dataset.from_pandas(df3)
d3_train,d3_test=split_data(d3)
d2_train=Dataset.from_dict(d3_train.to_dict())
d2_test=Dataset.from_dict(d3_test.to_dict())
d2_train=d2_train.map(lambda example: {"impression": None})
d2_test=d2_test.map(lambda example: {"impression": None})

dataset_train = concatenate_datasets([d1_train, d2_train,d3_train])
print(len(d1_train),len(d2_train),len(d3_train),len(dataset_train))
dataset_test = concatenate_datasets([d1_test, d2_test,d3_test])
print(len(d1_test),len(d2_test),len(d3_test),len(dataset_test))
dataset_train[0]

Map: 100%|██████████| 732/732 [00:00<00:00, 10624.66 examples/s]

1116 2927 2927 6970
279 732 732 1743


{'series_description': 'COR CISS T2 1MM MPR',
 'ground_truth_classification': 'T2',
 'index_col': 717,
 'ACCID': None,
 'Report ID': None,
 'Age': None,
 'Sex': None,
 'REPORT': None,
 'No Finding': None,
 'Enlarged Cardiom.': None,
 'Cardiomegaly': None,
 'Lung Lesion': None,
 'Lung Opacity': None,
 'Edema': None,
 'Consolidation': None,
 'Pneumonia': None,
 'Atelectasis': None,
 'Pneumothorax': None,
 'Pleural Effusion': None,
 'Pleural Other': None,
 'Fracture': None,
 'Support Devices': None,
 'impression': None,
 '__index_level_0__': None}

In [16]:
def filter_function(example):
    """
    Make sure only series harmonization data remains
    """
    return example["series_description"]  # Keep only rows where label is not 0
dataset_train = dataset_train.filter(filter_function)
dataset_test = dataset_test.filter(filter_function)

len(dataset_train),len(dataset_test)

Filter: 100%|██████████| 1743/1743 [00:00<00:00, 26866.41 examples/s]


(1116, 279)

In [17]:
from Trainer.preprocessing import preprocess_function_harmonize_series,preprocess_function_response,preprocess_function_impression
def preprocess_function(example):
    """
    Dynamically maps incoming dataset batches to their respective task prompts.
    This allows a single SLM to learn multi-task contextual switching.
    
    Args:
        example (dict): A batch of rows from the concatenated dataset.
    Returns:
        list: A list of fully formatted string prompts ready for tokenization.
    """
    output_text=[]
    #print(example)
    for i in range(len(example['series_description'])):
        if example['series_description'][i] is not None:
            output_text.append(preprocess_function_harmonize_series(example,i))
        elif example['impression'][i]:
            output_text.append(preprocess_function_impression(example,i))
        else:
            output_text.append(preprocess_function_response(example,i))

    return output_text

"\ndef preprocess_function(example):\n    \n    if example['series_description'] is not None:\n        return preprocess_function_harmonize_series(example,i)\n    elif example['impression']:\n        return preprocess_function_impression(example,i)\n    else:\n        return preprocess_function_response(example,i)\n"

## Training Config

In [18]:
import os
rank=32

batch_size = 4
num_workers = os.cpu_count()
epochs = 100
bf16 = False
fp16 = True
gradient_accumulation_steps = 8
context_length = 1024
learning_rate = 0.0004
model_name = 'facebook/opt-350m'
out_dir = '../models/opt_350m_1task_harmonize_rank'+str(rank)+'_new' ##CHANGE THIS
seed = 42


from peft import LoraConfig

config = LoraConfig(init_lora_weights=False,r=rank)

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m")
from peft import get_peft_model

lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()
lora_model

trainable params: 3,145,728 || all params: 334,342,144 || trainable%: 0.9409


PeftModel(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTSdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=32, bias=False)
                  )
     

In [19]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_fast=False
)

tokenizer.add_eos_token = True
tokenizer.bos_token = '<s>'

training_args = TrainingArguments(
    output_dir=f"{out_dir}/logs",
    eval_strategy='epoch',
    weight_decay=0.01,
    load_best_model_at_end=False,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    bf16=bf16,
    fp16=fp16,
    report_to='tensorboard',
    num_train_epochs=epochs,
    dataloader_num_workers=num_workers,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    lr_scheduler_type='constant',
    seed=seed
)

trainer = SFTTrainer(
    model=model,
    peft_config=config,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    max_seq_length=context_length,
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=preprocess_function
)

Map: 100%|██████████| 279/279 [00:00<00:00, 302.39 examples/s]
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [20]:
trainer.model

PeftModel(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTSdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=32, bias=False)
                  )
     

In [21]:
# Check example training data
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
dataloader = trainer.get_train_dataloader()
for i, sample in enumerate(dataloader):
    print(tokenizer.decode(sample['input_ids'][0]))
    print('#'*50)
    if i == 5:
        break

<s>### Instruction:
Harmonize sequence name.

### Input:
AX FLAIR (TxRx)

### Response: 
 {'AX FLAIR (TxRx)': 'T2/FLAIR'}</s>

##################################################
<s>### Instruction:
Harmonize sequence name.

### Input:
3Plane Loc

### Response: 
 {'3Plane Loc': 'Other'}</s>
<pad><pad><pad><pad><pad><pad>
##################################################
<s>### Instruction:
Harmonize sequence name.

### Input:
DTI_71dir_AP_STRAIGHT_TENSOR

### Response: 
 {'DTI_71dir_AP_STRAIGHT_TENSOR': 'Other'}</s>
<pad><pad><pad><pad><pad><pad><pad>
##################################################
<s>### Instruction:
Harmonize sequence name.

### Input:
SWI_Images

### Response: 
 {'SWI_Images': 'SWI'}</s>
<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
##################################################
<s>### Instruction:
Harmonize sequence name.

### Input:
SAG MPR 1x1mm

### Response: 
 {'SAG MPR 1x1mm': 'Other'}</s>

In [22]:
import warnings
warnings.filterwarnings('ignore')
history = trainer.train()
trainer.model.save_pretrained(out_dir+'/final_model')

Epoch,Training Loss,Validation Loss
0,1.870800,No log
1,0.930200,No log
2,0.784700,No log
3,0.688000,No log
4,0.617300,No log
5,0.566900,No log
6,0.529600,No log
7,0.496900,No log
8,0.469000,No log
9,0.444900,No log


## Eval

In [26]:
import torch
from peft import PeftModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

new_model=out_dir+'/final_model'
base_model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",
    device_map='cuda',
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained('facebook/opt-350m')
out_dir

'../models/opt_350m_1task_harmonize_rank32_new'

## Series Harmonization

In [27]:
template = "<s>### Instruction:\n{instruction}\n\n### Input:\n{text}\n\n### Response:"
instruction = 'Harmonize sequence name.'
import json
def summarize(i, tokenizer,  m, template):
    if dataset_test[i]['series_description'] is None:
        return None,None
    text = dataset_test[i]['series_description']
    idx=dataset_test[i]['index_col']
    with open("../harmonize_series/"+str(idx)+".json") as f:
            ground_truth = json.load(f)
            
    prompt = template.format(instruction=instruction, text=text)
    prompt_tokenized = tokenizer(
        prompt, 
        return_tensors='pt', 
        return_attention_mask=True
    ).to('cuda')
    output_tokenized = m.generate(
        **prompt_tokenized,
        eos_token_id=tokenizer.eos_token_id,
        max_length=len(prompt_tokenized['input_ids'][0])+500,
        repetition_penalty=1.1,
        temperature=0.8,
        top_k=40,
        top_p=0.1,
        do_sample=True,
        num_beams=5
    )
    
    answer = tokenizer.decode(token_ids=output_tokenized[0][len(prompt_tokenized[0]):]).strip().replace("'", "\"").split('}')[0]+'}'
    try:
        answer_dict=json.loads(answer)
    except:
        print(answer)
        return ground_truth,dict()
    return ground_truth,answer_dict

In [28]:
!pip install tqdm
from tqdm import tqdm
import numpy as np
c=0
acc=[]

eval_set=len(dataset_test)
for i in tqdm(range(eval_set)):
    ground_truth,answer_dict=summarize(i, tokenizer, model, template)
    if ground_truth is None:
        continue
    if ground_truth==answer_dict:
        acc.append(1)
    else:
        acc.append(0)
            
np.mean(acc)

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip


100%|██████████| 279/279 [01:17<00:00,  3.58it/s]


0.9713261648745519